# 🛡️ Real-Time Machine Learning Intrusion Detection System (IDS)
### OS Project — Self-Contained Jupyter Notebook

**Run each cell in order from top to bottom.**

| Phase | Description |
|-------|-------------|
| 1 | Install dependencies |
| 2 | Setup utilities & feature extractor |
| 3 | Data collection (real OS syscall traces) |
| 4 | Model training (Random Forest) |
| 5 | Real-time detection demo |
| 6 | Results & visualizations |

## 📦 Cell 1 — Install Dependencies

In [1]:
import subprocess, sys
packages = ["pandas", "numpy", "scikit-learn", "psutil",
            "matplotlib", "seaborn", "plotly", "flask", "flask-cors"]
for pkg in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"],
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"  OK: {pkg}")
    except Exception as e:
        print(f"  WARN {pkg}: {e}")
print("All packages processed!")


✅ All dependencies installed!


## ⚙️ Cell 2 — Utilities & Setup

In [2]:
import os, logging

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(), logging.FileHandler('ids_system.log')]
)
logger = logging.getLogger('IDS_Logger')
logger.info('IDS System Initialized.')
print('✅ Directories created: data/, models/')

2026-04-24 20:25:21,110 - INFO - IDS System Initialized.


✅ Directories created: data/, models/


## 🔬 Cell 3 — Feature Extractor (N-gram Syscall Vectorizer)

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

class SyscallFeatureExtractor:
    """
    Converts syscall sequences into N-gram feature vectors.
    Treats syscall sequences like text documents.
    """
    def __init__(self, ngram_range=(1, 2), max_features=300):
        self.vectorizer = CountVectorizer(
            ngram_range=ngram_range,
            max_features=max_features,
            token_pattern=r'(?u)\b\w+\b'
        )
        self.is_fitted = False

    def fit_transform(self, sequences):
        X = self.vectorizer.fit_transform(sequences)
        self.is_fitted = True
        return X.toarray()

    def transform(self, sequences):
        if not self.is_fitted:
            raise ValueError('Extractor must be fitted first.')
        return self.vectorizer.transform(sequences).toarray()

    def get_feature_names(self):
        return self.vectorizer.get_feature_names_out() if self.is_fitted else []

print('✅ SyscallFeatureExtractor class defined.')

✅ SyscallFeatureExtractor class defined.


## 🖥️ Cell 4 — Data Collection (Real OS Syscall Traces)

In [4]:
import sys, time, psutil, subprocess
import pandas as pd

def get_target_pid(target_names):
    """Find PID of a running browser or editor."""
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            if any(t in proc.info['name'].lower() for t in target_names):
                return proc.info['pid']
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return os.getpid()

def harvest_windows_proxy_syscalls(pid, duration=8):
    """Windows: map I/O counters -> syscall-like tokens."""
    syscalls = []
    start = time.time()
    try:
        proc = psutil.Process(pid)
    except:
        return ['poll', 'stat']
    while time.time() - start < duration:
        if not proc.is_running():
            break
        try:
            io1 = proc.io_counters()
            conn1 = len(proc.connections())
            time.sleep(0.2)
            io2 = proc.io_counters()
            conn2 = len(proc.connections())
            reads  = io2.read_count  - io1.read_count
            writes = io2.write_count - io1.write_count
            for _ in range(min(reads, 8)):  syscalls.append('read')
            for _ in range(min(writes, 8)): syscalls.append('write')
            if conn2 != conn1:             syscalls.extend(['socket', 'connect', 'bind'])
            if reads > 0 or writes > 0:    syscalls.extend(['open', 'mmap', 'fstat'])
            if not reads and not writes:   syscalls.extend(['poll', 'stat', 'gettimeofday'])
        except:
            time.sleep(0.5)
    return syscalls

def harvest_linux_strace(pid, duration=8):
    """Linux: use strace to capture actual syscalls."""
    syscalls = []
    try:
        proc = subprocess.Popen(
            ['strace', '-p', str(pid), '-q', '-e', 'trace=all'],
            stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True
        )
    except FileNotFoundError:
        return harvest_windows_proxy_syscalls(pid, duration)
    start = time.time()
    while time.time() - start < duration:
        try:
            line = proc.stderr.readline()
            if not line: break
            name = line.split('(')[0].strip()
            if name.isalnum():
                syscalls.append(name)
        except:
            continue
    proc.terminate()
    return syscalls

def collect_syscall_stream(pid, duration=8):
    """Cross-platform wrapper."""
    if sys.platform == 'win32':
        return harvest_windows_proxy_syscalls(pid, duration)
    return harvest_linux_strace(pid, duration)

def create_windows(syscalls, window_size=20):
    windows = []
    step = max(1, window_size // 2)
    for i in range(0, max(1, len(syscalls) - window_size + 1), step):
        windows.append(' '.join(syscalls[i:i + window_size]))
    return windows

print('✅ Data collection functions defined.')

✅ Data collection functions defined.


## 📊 Cell 5 — Generate Dataset (Runs Data Collection)

In [ ]:
import os
import subprocess

DATASET_PATH = 'data/dataset.csv'
DURATION = 8  # seconds per class — increase for richer data

data = []

# ── Normal traffic ──────────────────────────────────────────────
normal_pid = get_target_pid(['chrome', 'firefox', 'brave', 'code', 'explorer'])
logger.info(f'[1/2] Hooking normal activity on PID {normal_pid}...')
print(f'⏳ Collecting NORMAL syscalls from PID {normal_pid} for {DURATION}s...')
normal_calls = collect_syscall_stream(normal_pid, DURATION)
for w in create_windows(normal_calls):
    if w.strip():
        data.append({'sequence': w, 'label': 0})
print(f'   → Collected {len(normal_calls)} normal syscall tokens.')

# ── Malicious (simulator) ────────────────────────────────────────
# Inline simulator — no external file needed
import threading, socket

stop_sim = threading.Event()

def _sim_loop():
    test_file = os.path.join('data', 'fake_target.tmp')
    while not stop_sim.is_set():
        try:
            with open(test_file, 'w') as f:
                f.write('Encrypted Data ' * 50)
            os.chmod(test_file, 0o777)
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.settimeout(0.3)
                try: s.connect(('8.8.8.8', 53))
                except: pass
            if os.path.exists(test_file):
                os.unlink(test_file)
            time.sleep(0.05)
        except:
            time.sleep(0.1)

sim_thread = threading.Thread(target=_sim_loop, daemon=True)
sim_thread.start()
sim_pid = os.getpid()  # monitor our own process while sim runs

logger.info(f'[2/2] Collecting malicious traces (PID {sim_pid})...')
print(f'⏳ Collecting MALICIOUS syscalls for {DURATION}s...')
malware_calls = collect_syscall_stream(sim_pid, DURATION)
stop_sim.set()
for w in create_windows(malware_calls):
    if w.strip():
        data.append({'sequence': w, 'label': 1})
print(f'   → Collected {len(malware_calls)} malicious syscall tokens.')

# ── Validation Failsafe ──────────────────────────────────────────
df_val = pd.DataFrame(data)
needs_fallback = (
    df_val.empty or
    len(df_val) < 10 or
    df_val['label'].nunique() < 2 or
    df_val['label'].value_counts().min() < 3
)
if needs_fallback:
    logger.warning('OS permission limits hit — using synthetic fallback dataset.')
    print('⚠️  Using synthetic dataset (OS blocked kernel hooking).')
    data = []
    for _ in range(200):
        data.append({'sequence': 'read write open stat poll gettimeofday sched_yield mmap fstat read write open stat', 'label': 0})
        data.append({'sequence': 'execve clone ptrace kill chmod unlink socket bind mmap fstat execve clone ptrace kill', 'label': 1})

df = pd.DataFrame(data).sample(frac=1).reset_index(drop=True)
df.to_csv(DATASET_PATH, index=False)
logger.info(f'Dataset saved: {len(df)} rows.')
print(f'\n✅ Dataset ready → {DATASET_PATH}')
print(df['label'].value_counts().rename({0: 'Normal', 1: 'Malicious'}).to_string())
df.head()

2026-04-24 20:25:22,900 - INFO - [1/2] Hooking normal activity on PID 1456...


⏳ Collecting NORMAL syscalls from PID 1456 for 8s...


C:\Users\shrut\AppData\Local\Temp\ipykernel_9700\4061877596.py:27: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  conn1 = len(proc.connections())
C:\Users\shrut\AppData\Local\Temp\ipykernel_9700\4061877596.py:30: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  conn2 = len(proc.connections())


## 🤖 Cell 6 — Train Random Forest Model

In [ ]:
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

MODEL_PATH     = 'models/rf_model.pkl'
EXTRACTOR_PATH = 'models/extractor.pkl'

logger.info('Loading dataset...')
df = pd.read_csv(DATASET_PATH)
X_raw = df['sequence'].values
y     = df['label'].values

# Feature extraction
logger.info('Extracting N-gram features...')
extractor = SyscallFeatureExtractor(ngram_range=(1, 2), max_features=300)
X = extractor.fit_transform(X_raw)
print(f'Feature matrix shape: {X.shape}')

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train
logger.info('Training Random Forest...')
print('⏳ Training...')
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
cm   = confusion_matrix(y_test, y_pred)

print('\n📈 Model Performance Metrics')
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print(f'\nConfusion Matrix:\n{cm}')
print(f'\n{classification_report(y_test, y_pred, target_names=["Normal","Malicious"], zero_division=0)}')

# Save
with open(MODEL_PATH, 'wb') as f:     pickle.dump(model, f)
with open(EXTRACTOR_PATH, 'wb') as f: pickle.dump(extractor, f)
logger.info('Models saved.')
print(f'✅ Model saved → {MODEL_PATH}')
print(f'✅ Extractor saved → {EXTRACTOR_PATH}')

## 📊 Cell 7 — Visualizations

In [ ]:
import os

# Try matplotlib first; fall back to Plotly if blocked by Windows security
_USE_PLOTLY = False
try:
    import matplotlib
    matplotlib.use("Agg")  # non-interactive backend
    import matplotlib.pyplot as plt
    import seaborn as sns
except Exception as _e:
    print(f"matplotlib unavailable: {_e}")
    print("Switching to Plotly (no DLLs needed)...")
    _USE_PLOTLY = True

if not _USE_PLOTLY:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("IDS Model Analysis", fontsize=16, fontweight="bold")
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Normal","Malicious"],
                yticklabels=["Normal","Malicious"], ax=axes[0])
    axes[0].set_title("Confusion Matrix")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
    metrics = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1}
    colors  = ["#4CAF50","#2196F3","#FF9800","#9C27B0"]
    bars = axes[1].bar(metrics.keys(), metrics.values(), color=colors, edgecolor="white")
    axes[1].set_ylim(0, 1.1)
    axes[1].set_title("Performance Metrics")
    for bar, val in zip(bars, metrics.values()):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.02,
                     f"{val:.3f}", ha="center", fontweight="bold")
    counts = df["label"].value_counts()
    axes[2].pie(counts.values, labels=["Normal","Malicious"],
                colors=["#4CAF50","#F44336"], autopct="%1.1f%%",
                startangle=90, textprops={"fontsize": 12})
    axes[2].set_title("Dataset Distribution")
    plt.tight_layout()
    os.makedirs("data", exist_ok=True)
    plt.savefig("data/ids_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Chart saved to data/ids_analysis.png")
else:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=["Confusion Matrix","Performance Metrics","Dataset Distribution"],
        specs=[[{"type":"heatmap"},{"type":"bar"},{"type":"pie"}]]
    )
    # Confusion matrix
    fig.add_trace(go.Heatmap(
        z=cm, x=["Normal","Malicious"], y=["Normal","Malicious"],
        colorscale="Blues", showscale=False,
        text=cm, texttemplate="%{text}", textfont={"size":18}), row=1, col=1)
    # Metrics bar
    m_names  = ["Accuracy","Precision","Recall","F1"]
    m_values = [acc, prec, rec, f1]
    fig.add_trace(go.Bar(
        x=m_names, y=m_values,
        marker_color=["#4CAF50","#2196F3","#FF9800","#9C27B0"],
        text=[f"{v:.3f}" for v in m_values], textposition="outside"), row=1, col=2)
    # Dataset pie
    counts = df["label"].value_counts()
    fig.add_trace(go.Pie(
        labels=["Normal","Malicious"], values=counts.values,
        marker_colors=["#4CAF50","#F44336"]), row=1, col=3)
    fig.update_layout(
        title_text="IDS Model Analysis", height=420, showlegend=False,
        paper_bgcolor="#0d1117", plot_bgcolor="#0d1117", font=dict(color="white")
    )
    fig.show()
    print("Interactive Plotly chart shown (used because OS blocked matplotlib DLLs)")


## 🔍 Cell 8 — Real-Time Detection Demo

In [ ]:
import pickle, time
from collections import deque

# Load saved model & extractor
with open(MODEL_PATH, 'rb') as f:     loaded_model = pickle.load(f)
with open(EXTRACTOR_PATH, 'rb') as f: loaded_extractor = pickle.load(f)
loaded_extractor.is_fitted = True

alert_history = deque(maxlen=50)

def analyze_syscall_window(process_id, syscall_sequence, app_name='Unknown'):
    """Analyze a window of syscalls and return prediction."""
    try:
        features = loaded_extractor.transform([syscall_sequence])
        prob = loaded_model.predict_proba(features)[0]
        malicious_prob = prob[1]
        status = 'MALICIOUS' if malicious_prob > 0.6 else 'SAFE'
        event = {
            'timestamp':        time.strftime('%Y-%m-%d %H:%M:%S'),
            'pid':              process_id,
            'app_name':         app_name,
            'status':           status,
            'prediction':       status,
            'confidence':       round(malicious_prob * 100, 2),
            'sequence_preview': syscall_sequence[:40] + '...'
        }
        alert_history.appendleft(event)
        icon = '🚨' if status == 'MALICIOUS' else '✅'
        print(f"{icon} [{status}] PID {process_id} ({app_name}) — Confidence: {event['confidence']}%")
        return event
    except Exception as e:
        print(f'Error: {e}')
        return None

# ── Live demo: monitor THIS notebook process for a few seconds ──
print('🔍 Starting live detection demo (10 seconds)...\n')
my_pid = os.getpid()
demo_calls = collect_syscall_stream(my_pid, duration=10)
windows = create_windows(demo_calls, window_size=20)

if not windows:
    windows = [
        'read write open stat poll gettimeofday sched_yield mmap fstat read',
        'execve clone ptrace kill chmod unlink socket bind mmap fstat execve'
    ]

for i, w in enumerate(windows[:10]):
    analyze_syscall_window(my_pid, w, app_name='Jupyter_Notebook')

print(f'\n📋 Total events captured: {len(alert_history)}')

## 📋 Cell 9 — Events Summary Table

In [ ]:
import pandas as pd

if alert_history:
    events_df = pd.DataFrame(list(alert_history))[[
        'timestamp', 'pid', 'app_name', 'prediction', 'confidence', 'sequence_preview'
    ]]
    
    def color_prediction(val):
        color = '#ffcccc' if val == 'MALICIOUS' else '#ccffcc'
        return f'background-color: {color}'
    
    display(events_df.style.map(color_prediction, subset=['prediction']).set_caption('🛡️ IDS Detection Events'))
    print(f'\nSafe:      {sum(1 for e in alert_history if e["prediction"]=="SAFE")}')
    print(f'Malicious: {sum(1 for e in alert_history if e["prediction"]=="MALICIOUS")}')
else:
    print('No events captured yet. Run Cell 8 first.')

## 🧪 Cell 10 — Manual Test (Predict Any Syscall Sequence)
Change the sequence below and re-run to test your own inputs.

In [ ]:
# ── Modify these to test different scenarios ──────────────────────
TEST_CASES = [
    ('Normal Browser', 'read write open stat poll gettimeofday sched_yield mmap fstat read write open'),
    ('Malware Pattern', 'execve clone ptrace kill chmod unlink socket bind mmap fstat execve clone ptrace'),
    ('Mixed Activity',  'read write execve open stat clone poll gettimeofday chmod unlink socket bind'),
]

print('=' * 65)
print('         IDS MANUAL PREDICTION TEST')
print('=' * 65)

for label, seq in TEST_CASES:
    print(f'\n🔬 Test: {label}')
    print(f'   Sequence: {seq[:50]}...')
    analyze_syscall_window(9999, seq, app_name=label)

print('\n' + '=' * 65)

## 🌐 Cell 11 — Launch Live IDS Dashboard (Flask Server)
Run this cell **after** Cell 8. The live dashboard will appear embedded below.

In [ ]:
import subprocess, sys, threading, time, os

# Auto-install Flask if missing
for pkg in ["flask", "flask-cors"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

from flask import Flask, jsonify, render_template_string
try:
    from flask_cors import CORS
except ImportError:
    CORS = lambda app: app

# Read the existing index.html dashboard template
TEMPLATE_PATH = os.path.join(os.getcwd(), "templates", "index.html")
with open(TEMPLATE_PATH, "r", encoding="utf-8") as f:
    DASHBOARD_HTML = f.read()

flask_app = Flask(__name__)
CORS(flask_app)

# Use alert_history directly so new events appear live
_events = alert_history
_status = {"status": "ACTIVE DEFENSE (REAL-TIME)", "monitored_pids": [os.getpid()]}

@flask_app.route("/")
def index():
    return render_template_string(DASHBOARD_HTML)

@flask_app.route("/api/stats")
def stats():
    total     = len(_events)
    malicious = sum(1 for e in _events if e.get("prediction") == "MALICIOUS")
    return jsonify({"total": total, "safe": total - malicious,
                    "malicious": malicious, "status": _status["status"],
                    "monitored_pids": _status["monitored_pids"]})

@flask_app.route("/api/events")
def get_events():
    out = []
    for e in _events:
        out.append({"timestamp": e.get("timestamp",""),
                    "pid": e.get("pid", 0),
                    "app_name": e.get("app_name", "Unknown"),
                    "syscalls": e.get("syscalls", []),
                    "sequence_preview": e.get("sequence_preview", ""),
                    "prediction": e.get("prediction", "SAFE"),
                    "confidence": e.get("confidence", 0)})
    return jsonify(out)

# Start Flask in a background daemon thread
FLASK_PORT = 5050
def _run_flask():
    flask_app.run(host="127.0.0.1", port=FLASK_PORT, debug=False, use_reloader=False)

t = threading.Thread(target=_run_flask, daemon=True)
t.start()
time.sleep(1.5)   # give Flask a moment to start

# Display dashboard inside the notebook
from IPython.display import IFrame, display, HTML
link = f"http://127.0.0.1:{FLASK_PORT}"
display(HTML(f"<h3>🛡️ IDS Live Dashboard &mdash; <a href='{link}' target='_blank'>Open in new tab 🔗</a></h3>"))
display(IFrame(src=link, width="100%", height=720))
print(f"✅ Flask running at {link}")


## 🚨 Cell 12 — Inject Malicious Syscalls (Demo)
Run this cell to push **fake malicious events** into the live dashboard.
You will see red THREAT DETECTED cards appear in the website instantly.

In [ ]:
import time

MALICIOUS_SEQUENCES = [
    'execve clone ptrace kill chmod unlink socket bind mmap fstat execve clone',
    'ptrace kill chmod unlink socket bind execve clone ptrace kill chmod unlink',
    'chmod unlink socket bind mmap fstat ptrace kill chmod unlink execve clone',
    'socket bind execve clone ptrace kill chmod unlink socket bind mmap fstat',
    'kill chmod unlink socket bind mmap execve clone ptrace kill chmod unlink',
]

print('Injecting 5 malicious syscall windows...')
for i, seq in enumerate(MALICIOUS_SEQUENCES):
    analyze_syscall_window(
        process_id=1337 + i,
        syscall_sequence=seq,
        app_name=f'MALWARE_SIM_{i+1}'
    )
    time.sleep(0.3)

print(f'\n🚨 Done! Refresh the dashboard tab to see THREAT DETECTED alerts.')
print(f'Total events in feed: {len(alert_history)}')


## 🌐 Cell 13 — Share Dashboard Publicly via ngrok
Run this cell **after Cell 11** to get a **public link** anyone can open — no install needed on their side.

> **One-time setup:** Sign up free at https://ngrok.com, then copy your authtoken from the dashboard.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyngrok', '-q'])

from pyngrok import ngrok, conf
from IPython.display import display, HTML

# ── PASTE YOUR FREE NGROK AUTHTOKEN HERE ──────────────────────────────────
# Get it free from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = '3Co4D5Wp9WY89El0IagExShXFeg_6atX2MYPo7A2yT4wdgiLP'

if NGROK_TOKEN == '3Co4D5Wp9WY89El0IagExShXFeg_6atX2MYPo7A2yT4wdgiLP':
    print('⚠️  Please sign up FREE at https://ngrok.com and paste your authtoken above!')
    print('   Then re-run this cell.')
else:
    # Set token and open tunnel to Flask (port 5050)
    conf.get_default().auth_token = NGROK_TOKEN

    # Kill any existing tunnels first
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)

    public_url = ngrok.connect(5050)
    link = public_url.public_url

    display(HTML(f'''
    <div style="background:#1a1a2e;padding:20px;border-radius:12px;border:2px solid #00f0ff;margin:10px 0;font-family:monospace">
      <h2 style="color:#00f0ff;margin:0 0 10px 0">🛡️ IDS Dashboard — Public Link</h2>
      <p style="color:#e2e8f0;font-size:18px;margin:0">
        🔗 <a href="{link}" target="_blank" style="color:#00f0ff">{link}</a>
      </p>
      <p style="color:#64748b;margin:8px 0 0 0;font-size:13px">✔ Share this link with anyone — it works immediately, no install needed.</p>
    </div>
    '''))
    print(f'Public URL: {link}')
    print('Share this link! It stays active as long as this notebook is running.')


---
## ✅ Project Complete!

| File | Purpose |
|------|----------|
| `data/dataset.csv` | Collected syscall dataset |
| `models/rf_model.pkl` | Trained Random Forest model |
| `models/extractor.pkl` | N-gram feature extractor |
| `data/ids_analysis.png` | Performance charts |
| `ids_system.log` | Full system log |

> **To run on another laptop:** Just open this `.ipynb` file in Jupyter and run **Kernel → Restart & Run All**. Cell 1 installs all required packages automatically.